### Data Modeling

### Create Flag Parameter

In [0]:
dbutils.widgets.text('incremental_flag', '0')

In [0]:
incremental_flag = dbutils.widgets.get('incremental_flag')
print(incremental_flag)

### Creating Dimensions Model

#### Fetch Related Columns

In [0]:
df_src = spark.sql('''
                   SELECT DISTINCT(Model_ID) as Model_ID, model_category 
                   FROM parquet.`abfss://silver@cacdyamadls.dfs.core.windows.net/carsales`
                   ''')

#df_src.display()

#### Dim Model Sink - Initial and Incremental Load

In [0]:
if spark.catalog.tableExists('cars_catalog.gold.dim_model') :
    df_sink = spark.sql('''
    SELECT dim_model_key, Model_ID, model_Category
    FROM parquet.`abfss://silver@cacdyamadls.dfs.core.windows.net/carsales`
    ''')
else :
    df_sink = spark.sql('''
    SELECT 1 as dim_model_key, Model_ID, model_Category
    FROM parquet.`abfss://silver@cacdyamadls.dfs.core.windows.net/carsales`
    WHERE 1=0
    ''')
    
                        
#df_sink.display()

### Filtering new records and old records

In [0]:
df_filter = df_src.join(df_sink, df_src['Model_ID'] == df_sink['Model_ID'], 'left').select(df_src['Model_ID'], df_src['model_category'], df_sink['dim_model_key'])

In [0]:
df_filter.display()